# Fine-Tune Live Segmentation Model

Fine-tune the strong hair segmentation checkpoint on webcam/live-domain frames so the live mask behaves better under real camera framing, lighting, and pose.

In [ ]:
from pathlib import Path
import sys
import time
import random

import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

def format_seconds(seconds: float) -> str:
    total_seconds = max(0, int(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f'{hours:d}:{minutes:02d}:{secs:02d}'
    return f'{minutes:02d}:{secs:02d}'

def resolve_training_device(torch_module, require_gpu: bool = True) -> str:
    if torch_module.cuda.is_available():
        return 'cuda'
    if require_gpu:
        raise RuntimeError(
            'CUDA GPU is required for this training run, but the current PyTorch build does not have CUDA available. '
            'Install a CUDA-enabled PyTorch build into .venv and restart the notebook kernel.'
        )
    return 'cpu'

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

import torch
from torch import nn
from torch.utils.data import DataLoader

torch.backends.cudnn.benchmark = True

from app.ml.datasets import HairSegmentationDataset, read_jsonl_manifest
from app.ml.hair_segmentation_model import build_segmentation_model
from app.ml.metrics import dice_coefficient_from_logits, iou_from_logits, pixel_accuracy_from_logits
from app.ml.live_segmentation import LIVE_SEGMENTATION_DATASET_DIR
from app.ml.transforms import ComposeImageMaskTransforms, RandomHorizontalFlipPair, ResizeImageAndMask

PROJECT_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon')

In [ ]:
REQUIRE_GPU = True
DEVICE = resolve_training_device(torch, require_gpu=REQUIRE_GPU)
BASE_DATASET_DIR = PROJECT_ROOT / 'backend' / 'data' / 'datasets' / 'hair_segmentation'
LIVE_DATASET_DIR = LIVE_SEGMENTATION_DATASET_DIR
BASE_CHECKPOINT_PATH = PROJECT_ROOT / 'backend' / 'checkpoints' / 'hair_segmentation' / 'v2_unet_product.pt'
CHECKPOINT_PATH = PROJECT_ROOT / 'backend' / 'checkpoints' / 'hair_segmentation' / 'v3_unet_live_finetuned.pt'

IMAGE_SIZE = (256, 256)
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
EPOCHS = 6
BASE_TRAIN_SAMPLE_LIMIT = 4000
BASE_VAL_SAMPLE_LIMIT = 800
LIVE_REPEAT_FACTOR = 10
SEED = 42

print(pd.Series({
    'device': DEVICE,
    'base_checkpoint': str(BASE_CHECKPOINT_PATH),
    'live_dataset_dir': str(LIVE_DATASET_DIR),
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
}))

device                                                           cuda
base_checkpoint     D:\Projects\Personal Projects\Hairstyle Recomm...
live_dataset_dir    D:\Projects\Personal Projects\Hairstyle Recomm...
batch_size                                                          8
epochs                                                              6
dtype: object


In [ ]:
base_train_records = read_jsonl_manifest(BASE_DATASET_DIR / 'train.jsonl')
base_val_records = read_jsonl_manifest(BASE_DATASET_DIR / 'val.jsonl')
live_train_records = read_jsonl_manifest(LIVE_DATASET_DIR / 'train.jsonl')
live_val_records = read_jsonl_manifest(LIVE_DATASET_DIR / 'val.jsonl')
live_test_records = read_jsonl_manifest(LIVE_DATASET_DIR / 'test.jsonl')

if not live_train_records:
    raise RuntimeError('Live adaptation train.jsonl is empty. Run the live dataset preparation notebook first.')

rng = random.Random(SEED)
base_train_pool = list(base_train_records)
base_val_pool = list(base_val_records)
rng.shuffle(base_train_pool)
rng.shuffle(base_val_pool)

base_train_subset_size = min(len(base_train_pool), max(BASE_TRAIN_SAMPLE_LIMIT, len(live_train_records) * 8))
base_val_subset_size = min(len(base_val_pool), max(BASE_VAL_SAMPLE_LIMIT, len(live_val_records) * 6 if live_val_records else 0))

base_train_subset = base_train_pool[:base_train_subset_size]
base_val_subset = base_val_pool[:base_val_subset_size]
live_train_augmented = live_train_records * LIVE_REPEAT_FACTOR

train_records = base_train_subset + live_train_augmented
val_records = base_val_subset + live_val_records

train_mix_summary = pd.DataFrame([
    {'split': 'train', 'base_records': len(base_train_subset), 'live_records_effective': len(live_train_augmented), 'total_records': len(train_records)},
    {'split': 'val', 'base_records': len(base_val_subset), 'live_records_effective': len(live_val_records), 'total_records': len(val_records)},
    {'split': 'test_live_only', 'base_records': 0, 'live_records_effective': len(live_test_records), 'total_records': len(live_test_records)},
])
display(train_mix_summary)

RuntimeError: Live adaptation train.jsonl is empty. Run the live dataset preparation notebook first.

In [ ]:
train_transform = ComposeImageMaskTransforms([
    RandomHorizontalFlipPair(probability=0.5),
    ResizeImageAndMask(IMAGE_SIZE),
])
val_transform = ResizeImageAndMask(IMAGE_SIZE)

train_dataset = HairSegmentationDataset(train_records, transform=train_transform)
val_dataset = HairSegmentationDataset(val_records, transform=val_transform)
live_test_dataset = HairSegmentationDataset(live_test_records, transform=val_transform) if live_test_records else None

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
live_test_loader = DataLoader(live_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True) if live_test_dataset is not None else None

payload = torch.load(BASE_CHECKPOINT_PATH, map_location=DEVICE)
model = build_segmentation_model(model_name='unet', base_channels=32).to(DEVICE)
model.load_state_dict(payload['model_state_dict'])
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss()

print('Loaded base checkpoint:', BASE_CHECKPOINT_PATH)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / max(len(loader), 1)

@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    logits_batches = []
    mask_batches = []
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, masks)
        running_loss += loss.item()
        logits_batches.append(logits)
        mask_batches.append(masks)

    if logits_batches:
        all_logits = torch.cat(logits_batches, dim=0)
        all_masks = torch.cat(mask_batches, dim=0)
        return {
            'loss': running_loss / max(len(loader), 1),
            'dice': dice_coefficient_from_logits(all_logits, all_masks),
            'iou': iou_from_logits(all_logits, all_masks),
            'pixel_accuracy': pixel_accuracy_from_logits(all_logits, all_masks),
        }

    return {'loss': 0.0, 'dice': 0.0, 'iou': 0.0, 'pixel_accuracy': 0.0}

In [ ]:
history = []
best_live_dice = -1.0
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    print(f'Running epoch {epoch}/{EPOCHS} [live-finetune]...')
    epoch_start = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_metrics = evaluate(model, val_loader, criterion, DEVICE)
    live_test_metrics = evaluate(model, live_test_loader, criterion, DEVICE) if live_test_loader is not None else None
    epoch_seconds = time.perf_counter() - epoch_start

    live_dice = live_test_metrics['dice'] if live_test_metrics is not None else val_metrics['dice']
    if live_dice > best_live_dice:
        best_live_dice = live_dice
        best_epoch = epoch

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_metrics['loss'],
        'val_dice': val_metrics['dice'],
        'val_iou': val_metrics['iou'],
        'val_pixel_accuracy': val_metrics['pixel_accuracy'],
        'live_test_loss': None if live_test_metrics is None else live_test_metrics['loss'],
        'live_test_dice': None if live_test_metrics is None else live_test_metrics['dice'],
        'live_test_iou': None if live_test_metrics is None else live_test_metrics['iou'],
        'live_test_pixel_accuracy': None if live_test_metrics is None else live_test_metrics['pixel_accuracy'],
        'epoch_time': format_seconds(epoch_seconds),
        'best_epoch_so_far': best_epoch,
        'best_live_dice_so_far': round(best_live_dice, 4),
    }
    history.append(row)
    print('Completed epoch', epoch)
    print(row)

history_df = pd.DataFrame(history)
display(history_df)

checkpoint_payload = {
    'model_state_dict': model.state_dict(),
    'history': history,
    'last_epoch': EPOCHS,
    'best_epoch': best_epoch,
    'best_live_dice': best_live_dice,
    'base_checkpoint_path': str(BASE_CHECKPOINT_PATH),
    'live_dataset_dir': str(LIVE_DATASET_DIR),
    'train_mix_summary': train_mix_summary.to_dict(orient='records'),
}
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(checkpoint_payload, CHECKPOINT_PATH)
print('Saved:', CHECKPOINT_PATH)